# สกัดพิกัดสถานีรถไฟฟ้าและทางเข้าออกผ่าน Overpass API
สมุดโน้ตเล่มนี้ถูกปรับปรุงให้ดึงข้อมูลสถานีรถไฟฟ้าและทางเข้าออกสถานีผ่าน **Overpass API** (ดึงผ่านอินเทอร์เน็ตโดยตรง) ทำให้ไม่ต้องโหลดและแกะไฟล์แผนที่ดิบ `osm.pbf` ขนาด 324MB ในเครื่อง ซึ่งจะช่วยย่นระยะเวลาการประมวลผลจากหลายนาทีเหลือเพียง **1-3 วินาที** เท่านั้น

In [ ]:
import pandas as pd
import requests
import os
from pathlib import Path

# ค้นหาตำแหน่งโฟลเดอร์หลักของโปรเจกต์
NOTEBOOK_DIR = Path(os.getcwd())
BASE_DIR = NOTEBOOK_DIR.parent.parent # เลื่อนขึ้นไปที่ ZoneVision/data-pipeline

print(f"Notebook Directory: {NOTEBOOK_DIR}")
print(f"Base Directory: {BASE_DIR}")

In [ ]:
# โหลดตั้งค่าขอบเขตรอยต่อกรุงเทพฯ (BBox) จากคอนฟิก
import json
config_path = BASE_DIR / "config.json"
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

bkk_bbox = config.get("bkk_bbox", [100.30, 13.45, 100.95, 13.95])

# Overpass API ใช้รูปแบบกล่องขอบเขต Bounding Box: (min_lat, min_lon, max_lat, max_lon)
# bkk_bbox เก็บค่า: [min_lon, min_lat, max_lon, max_lat]
bbox_str = f"{bkk_bbox[1]},{bkk_bbox[0]},{bkk_bbox[3]},{bkk_bbox[2]}"

print(f"Bangkok BBox for Overpass API: {bbox_str}")

In [ ]:
# 1. ดึงข้อมูลสถานีรถไฟฟ้า (railway=station) ผ่าน Overpass API
print("กำลังดาวน์โหลดพิกัดสถานีรถไฟฟ้าจาก Overpass API...")
overpass_url = "http://overpass-api.de/api/interpreter"
query_stations = f"""
[out:json][timeout:25];
(
  node["railway"="station"]({bbox_str});
  way["railway"="station"]({bbox_str});
  relation["railway"="station"]({bbox_str});
);
out center;
"""

headers = {
    'User-Agent': 'ZoneVisionSeniorProject/1.0 (contact: naeiger@example.com)'
}

try:
    response = requests.post(overpass_url, data={'data': query_stations}, headers=headers, timeout=30)
    if response.status_code == 200:
        data = response.json()
        elements = data.get('elements', [])
        print(f"🎉 ดึงข้อมูลสำเร็จ! พบรายการข้อมูลสถานี: {len(elements)} รายการ")
        
        stations = []
        for el in elements:
            lat = el.get('lat') or el.get('center', {}).get('lat')
            lon = el.get('lon') or el.get('center', {}).get('lon')
            tags = el.get('tags', {})
            name = tags.get('name') or tags.get('name:en') or tags.get('name:th')
            railway = tags.get('railway')
            operator = tags.get('operator')
            subway = tags.get('subway')
            light_rail = tags.get('light_rail')
            
            stations.append({
                'name': name,
                'railway': railway,
                'latitude': lat,
                'longitude': lon,
                'operator': operator,
                'subway': subway,
                'light_rail': light_rail
            })
            
        stations_df = pd.DataFrame(stations)
        # คลีนข้อมูลลบค่าว่าง
        stations_df['name'] = stations_df['name'].fillna('Unnamed Station')
        stations_clean = stations_df.dropna(subset=['latitude', 'longitude']).drop_duplicates(subset=['latitude', 'longitude'])
        
        print(f"คงเหลือข้อมูลสถานีที่ไม่ซ้ำซ้อนกัน: {len(stations_clean)} สถานี")
        print(stations_clean.head(10))
    else:
        print(f"❌ การดึงข้อมูลล้มเหลว: HTTP Code {response.status_code}")
except Exception as e:
    print(f"⚠️ เกิดข้อผิดพลาดในการดึงข้อมูล: {str(e)}")

In [ ]:
# 2. ดึงข้อมูลประตูทางเข้าออกสถานี (railway=subway_entrance) ผ่าน Overpass API
print("กำลังดาวน์โหลดพิกัดประตูทางออกรถไฟฟ้าจาก Overpass API...")
query_entrances = f"""
[out:json][timeout:25];
(
  node["railway"="subway_entrance"]({bbox_str});
);
out body;
"""

headers = {
    'User-Agent': 'ZoneVisionSeniorProject/1.0 (contact: naeiger@example.com)'
}

try:
    response = requests.post(overpass_url, data={'data': query_entrances}, headers=headers, timeout=30)
    if response.status_code == 200:
        data = response.json()
        elements = data.get('elements', [])
        print(f"🎉 ดึงข้อมูลสำเร็จ! พบรายการข้อมูลประตูทางออก: {len(elements)} รายการ")
        
        entrances = []
        for el in elements:
            lat = el.get('lat')
            lon = el.get('lon')
            tags = el.get('tags', {})
            name = tags.get('name') or tags.get('name:en')
            ref = tags.get('ref')
            railway = tags.get('railway')
            operator = tags.get('operator')
            
            entrances.append({
                'name': name,
                'ref': ref,
                'railway': railway,
                'latitude': lat,
                'longitude': lon,
                'operator': operator
            })
            
        entrances_df = pd.DataFrame(entrances)
        entrances_clean = entrances_df.dropna(subset=['latitude', 'longitude']).drop_duplicates(subset=['latitude', 'longitude'])
        
        print(f"คงเหลือข้อมูลทางออกสถานีที่ไม่ซ้ำกัน: {len(entrances_clean)} จุด")
        print(entrances_clean.head(10))
    else:
        print(f"❌ การดึงข้อมูลล้มเหลว: HTTP Code {response.status_code}")
except Exception as e:
    print(f"⚠️ เกิดข้อผิดพลาดในการดึงข้อมูล: {str(e)}")

In [ ]:
# 3. บันทึกผลลัพธ์ลงคลังข้อมูลชั่วคราว (Save to Interim Data)
output_dir = BASE_DIR / "data" / "interim"
os.makedirs(output_dir, exist_ok=True)

if 'stations_clean' in locals() and len(stations_clean) > 0:
    stations_file = output_dir / "bangkok_transit_stations.json"
    stations_clean.to_json(stations_file, orient='records', force_ascii=False, indent=4)
    print(f"💾 บันทึกไฟล์พิกัดสถานีรถไฟฟ้าระดับอินเตอร์ริมสำเร็จ: {stations_file}")

if 'entrances_clean' in locals() and len(entrances_clean) > 0:
    entrances_file = output_dir / "bangkok_transit_entrances.json"
    entrances_clean.to_json(entrances_file, orient='records', force_ascii=False, indent=4)
    print(f"💾 บันทึกไฟล์พิกัดประตูทางออกสถานีรถไฟฟ้าระดับอินเตอร์ริมสำเร็จ: {entrances_file}")
    
print("✅ เสร็จสมบูรณ์!")